# Chapter 9 — Variational and hybrid circuits with `qcirclab`

This notebook mirrors the original Chapter 9 examples, replacing Qiskit-specific constructs with the lightweight `qcirclab` circuit layer and small NumPy utilities.

In [1]:
# In a fresh environment, install qcirclab from the repository used in the book:
!pip install git+https://github.com/2forts/qcirclab_repo.git

  Cloning https://github.com/2forts/qcirclab_repo.git to /tmp/pip-req-build-p61_p7k3
  Running command git clone --filter=blob:none --quiet https://github.com/2forts/qcirclab_repo.git /tmp/pip-req-build-p61_p7k3
  Resolved https://github.com/2forts/qcirclab_repo.git to commit 4100251a2fe7c12a0848fafd5015d181014a60ab
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for qcirclab: filename=qcirclab-0.1.0-py3-none-any.whl size=18785 sha256=ae6845a4e0d854ab721a34f253615e6e2d8abac4a16f41b1aaa5c361f7032172
  Stored in directory: /tmp/pip-ephem-wheel-cache-st7ufon9/wheels/a8/2a/03/88707ddacd99a1ac6881e5e22a00b6f637d8cf5b9592d51074
Successfully built qcirclab


In [2]:
import numpy as np

from qcirclab import Circuit, basic_metrics
import qcirclab.gates as qg


In [3]:
# Shared utilities for this chapter

PAULI_MATS = {
    "I": np.eye(2, dtype=complex),
    "X": qg.X,
    "Y": qg.Y,
    "Z": qg.Z,
}


def kron_all(mats):
    out = np.array([[1.0]], dtype=complex)
    for m in mats:
        out = np.kron(out, m)
    return out


def pauli_matrix(label: str) -> np.ndarray:
    # Return the matrix for a Pauli string in q0...q(n-1) order.
    return kron_all([PAULI_MATS[ch] for ch in label])


def expectation_state(state: np.ndarray, operator: np.ndarray) -> float:
    return float(np.real(np.vdot(state, operator @ state)))


def expectation_pauli(qc: Circuit, label: str) -> float:
    state = qc.statevector()
    return expectation_state(state, pauli_matrix(label))


def expectation_hamiltonian(qc: Circuit, terms) -> float:
    # terms is a list of (pauli_label, coefficient).
    state = qc.statevector()
    value = 0.0
    for label, coeff in terms:
        value += coeff * expectation_state(state, pauli_matrix(label))
    return float(np.real(value))


def reduced_density_matrix(state: np.ndarray, n_qubits: int, keep):
    # Partial trace, keeping the qubits listed in keep.
    keep = tuple(keep)
    traced = tuple(q for q in range(n_qubits) if q not in keep)
    rho = np.outer(state, state.conj()).reshape([2] * (2 * n_qubits))
    current_n = n_qubits
    # Trace in descending order so axis indices remain valid.
    for q in sorted(traced, reverse=True):
        rho = np.trace(rho, axis1=q, axis2=q + current_n)
        current_n -= 1
    dim = 2 ** len(keep)
    return rho.reshape((dim, dim))


def entanglement_entropy_one_qubit(state: np.ndarray, n_qubits: int, qubit: int) -> float:
    rho = reduced_density_matrix(state, n_qubits, keep=[qubit])
    evals = np.linalg.eigvalsh(rho)
    evals = np.clip(evals, 0.0, 1.0)
    return float(-np.sum(evals * np.log(evals + 1e-12)))


def rxx_matrix(theta: float) -> np.ndarray:
    return np.cos(theta / 2) * np.eye(4) - 1j * np.sin(theta / 2) * np.kron(qg.X, qg.X)


def rzz_matrix(theta: float) -> np.ndarray:
    return np.diag([
        np.exp(-1j * theta / 2),
        np.exp(1j * theta / 2),
        np.exp(1j * theta / 2),
        np.exp(-1j * theta / 2),
    ]).astype(complex)


def add_rxx(qc: Circuit, theta: float, q0: int, q1: int) -> Circuit:
    return qc.unitary(rxx_matrix(theta), [q0, q1], name="rxx")


def add_rzz(qc: Circuit, theta: float, q0: int, q1: int) -> Circuit:
    return qc.unitary(rzz_matrix(theta), [q0, q1], name="rzz")


def print_basic_metrics(name: str, qc: Circuit):
    print(f"=== {name} ===")
    print(basic_metrics(qc))


# Subsection **9.2.1 Formal definition and structure**

In [4]:
# A parameterized circuit is represented here as a builder function.
# Passing numerical parameters returns an ordinary qcirclab Circuit.

def pqc_2q(theta0: float, theta1: float) -> Circuit:
    qc = Circuit(2)
    qc.ry(theta0, 0)
    qc.cx(0, 1)
    qc.rz(theta1, 1)
    qc.cx(1, 0)
    return qc

qc = pqc_2q(theta0=0.3, theta1=0.8)
print(qc.draw())


q0: |0>[ RY]─●────  ─X─
q1: |0>───  ─X─[ RZ]─●─


# Subsection **9.2.2 Expressibility, entangling capacity, and design criteria**

In [5]:
# Define a simple 2-layer ansatz on 3 qubits.

def layered_ansatz_3q(params) -> Circuit:
    qc = Circuit(3)
    qc.ry(params[0], 0)
    qc.ry(params[1], 1)
    qc.ry(params[2], 2)
    qc.cx(0, 1)
    qc.cx(1, 2)
    qc.ry(params[3], 0)
    qc.ry(params[4], 1)
    qc.ry(params[5], 2)
    return qc

samples = np.random.uniform(0, 2 * np.pi, size=6)
qc_bound = layered_ansatz_3q(samples)
state = qc_bound.statevector()

entropy = entanglement_entropy_one_qubit(state, n_qubits=3, qubit=0)
print("Entanglement entropy of qubit 0:", entropy)


Entanglement entropy of qubit 0: 0.06055989037246921


# Subsection **9.2.3 Common parameterized gate families**

In [6]:
theta = 0.4
phi = 0.9

qc = Circuit(2)

# Single-qubit rotations
qc.rx(theta, 0)
qc.ry(phi, 1)
qc.rz(theta, 1)

# Two-qubit parameterized interactions implemented as custom unitaries
add_rxx(qc, phi, 0, 1)
add_rzz(qc, theta, 0, 1)

# Controlled parameterized phase
qc.cp(phi, 0, 1)

print(qc.draw())


q0: |0>[ RX]───  ───  [RXX][RZZ]─●─
q1: |0>───  [ RY][ RZ][RXX][RZZ][ CP]


# Subsection **9.3.1 Ansatze families: Hardware-efficient, problem-inspired, symmetry-preserving**

In [7]:
def hardware_efficient_ansatz(n_qubits: int, layers: int, params) -> Circuit:
    qc = Circuit(n_qubits)
    k = 0
    for _ in range(layers):
        for q in range(n_qubits):
            qc.ry(params[k], q)
            k += 1
        for q in range(n_qubits - 1):
            qc.cx(q, q + 1)
    return qc

n_qubits = 3
layers = 2
theta = np.linspace(0.1, 0.6, n_qubits * layers)
qc = hardware_efficient_ansatz(n_qubits, layers, theta)
print(qc.draw())


q0: |0>[ RY]───  ───  ─●────[ RY]───  ───  ─●────
q1: |0>───  [ RY]───  ─X──●────  [ RY]───  ─X──●─
q2: |0>───  ───  [ RY]────X────  ───  [ RY]────X─


In [8]:
# A small EfficientSU2-like ansatz built manually with qcirclab.

def efficient_su2_like(num_qubits: int, reps: int, params) -> Circuit:
    qc = Circuit(num_qubits)
    k = 0
    for _ in range(reps):
        for q in range(num_qubits):
            qc.ry(params[k], q)
            k += 1
            qc.rz(params[k], q)
            k += 1
        for q in range(num_qubits - 1):
            qc.cx(q, q + 1)
    return qc

params = np.linspace(0.1, 1.2, 3 * 2 * 2)
ansatz = efficient_su2_like(num_qubits=3, reps=2, params=params)
print(ansatz.draw())


q0: |0>[ RY][ RZ]───  ───  ───  ───  ─●────[ RY][ RZ]───  ───  ───  ───  ─●────
q1: |0>───  ───  [ RY][ RZ]───  ───  ─X──●────  ───  [ RY][ RZ]───  ───  ─X──●─
q2: |0>───  ───  ───  ───  [ RY][ RZ]────X────  ───  ───  ───  [ RY][ RZ]────X─


# Subsection **9.3.2 Cost functions and observables**

In [9]:
# Parameterized single-qubit ansatz and observable H = 0.7 Z + 0.3 X

def one_qubit_ansatz(theta: float) -> Circuit:
    qc = Circuit(1)
    qc.ry(theta, 0)
    return qc

H = [("Z", 0.7), ("X", 0.3)]
value = np.pi / 4
expect = expectation_hamiltonian(one_qubit_ansatz(value), H)
print("Cost value:", expect)


Cost value: 0.7071067811865475


# Subsection **9.3.3 Gradient computation: Parameter-shift and other methods**

In [10]:
# Observable Z

def expectation(theta_value: float) -> float:
    return expectation_pauli(one_qubit_ansatz(theta_value), "Z")


def parameter_shift(theta_value: float) -> float:
    shift = np.pi / 2
    return 0.5 * (expectation(theta_value + shift) - expectation(theta_value - shift))

theta_val = np.pi / 4
grad_val = parameter_shift(theta_val)
print("Gradient:", grad_val)


Gradient: -0.7071067811865475


# Subsection **9.3.4 Training challenges: Barren plateaus and landscape issues**

In [11]:
# Two-layer hardware-efficient ansatz on 4 qubits

def barren_ansatz(params) -> Circuit:
    qc = Circuit(4)
    k = 0
    for _ in range(2):
        for q in range(4):
            qc.ry(params[k], q)
            k += 1
        for q in range(3):
            qc.cx(q, q + 1)
    return qc


def expectation_global(params) -> float:
    return expectation_pauli(barren_ansatz(params), "ZZZZ")


def gradient(params):
    grads = np.zeros_like(params)
    shift = np.pi / 2
    for i in range(len(params)):
        shifted_plus = params.copy()
        shifted_minus = params.copy()
        shifted_plus[i] += shift
        shifted_minus[i] -= shift
        grads[i] = 0.5 * (
            expectation_global(shifted_plus) - expectation_global(shifted_minus)
        )
    return grads

params = np.random.uniform(0, 2 * np.pi, size=8)
grads = gradient(params)
print("Gradient magnitudes:", np.abs(grads))


Gradient magnitudes: [2.23721974e-01 3.21259446e-01 3.43750903e-02 2.53897085e-01
 1.24900090e-16 1.13108767e-01 6.93889390e-17 4.20131741e-01]


# Subsection **9.4.1 Iterative optimization loops**

In [12]:
def cost(theta_value: float) -> float:
    return expectation_pauli(one_qubit_ansatz(theta_value), "Z")


def grad(theta_value: float) -> float:
    shift = np.pi / 2
    return 0.5 * (cost(theta_value + shift) - cost(theta_value - shift))

theta_k = 1.0
eta = 0.1
max_iters = 30

for k in range(max_iters):
    c_val = cost(theta_k)
    g_val = grad(theta_k)
    print(f"Iter {k:2d}: theta = {theta_k:.4f}, C(theta) = {c_val:.6f}, grad = {g_val:.6f}")
    theta_k = theta_k - eta * g_val


Iter  0: theta = 1.0000, C(theta) = 0.540302, grad = -0.841471
Iter  1: theta = 1.0841, C(theta) = 0.467667, grad = -0.883905
Iter  2: theta = 1.1725, C(theta) = 0.387814, grad = -0.921738
Iter  3: theta = 1.2647, C(theta) = 0.301328, grad = -0.953521
Iter  4: theta = 1.3601, C(theta) = 0.209177, grad = -0.977878
Iter  5: theta = 1.4579, C(theta) = 0.112705, grad = -0.993628
Iter  6: theta = 1.5572, C(theta) = 0.013582, grad = -0.999908
Iter  7: theta = 1.6572, C(theta) = -0.086301, grad = -0.996269
Iter  8: theta = 1.7568, C(theta) = -0.184964, grad = -0.982745
Iter  9: theta = 1.8551, C(theta) = -0.280495, grad = -0.959855
Iter 10: theta = 1.9511, C(theta) = -0.371195, grad = -0.928555
Iter 11: theta = 2.0439, C(theta) = -0.455693, grad = -0.890137
Iter 12: theta = 2.1330, C(theta) = -0.533019, grad = -0.846103
Iter 13: theta = 2.2176, C(theta) = -0.602616, grad = -0.798031
Iter 14: theta = 2.2974, C(theta) = -0.664316, grad = -0.747452
Iter 15: theta = 2.3721, C(theta) = -0.718277, 

# Subsection **9.4.2 Classical optimizers for variational algorithms**

In [13]:
def cost_wrapper(theta_array):
    theta_value = theta_array[0]
    return expectation_pauli(one_qubit_ansatz(theta_value), "Z")


def naive_search(x0, max_iters=50, step=0.1):
    x = np.array(x0, dtype=float)
    for k in range(max_iters):
        c_val = cost_wrapper(x)
        left = cost_wrapper(x - step)
        right = cost_wrapper(x + step)
        if left < c_val and left <= right:
            x = x - step
        elif right < c_val:
            x = x + step
        else:
            step *= 0.5
        print(f"Iter {k:2d}: x = {x[0]:.4f}, C(x) = {c_val:.6f}")
    return x

x0 = [0.7]
opt_params = naive_search(x0)
print("Optimized parameter:", opt_params[0])


Iter  0: x = 0.8000, C(x) = 0.764842
Iter  1: x = 0.9000, C(x) = 0.696707
Iter  2: x = 1.0000, C(x) = 0.621610
Iter  3: x = 1.1000, C(x) = 0.540302
Iter  4: x = 1.2000, C(x) = 0.453596
Iter  5: x = 1.3000, C(x) = 0.362358
Iter  6: x = 1.4000, C(x) = 0.267499
Iter  7: x = 1.5000, C(x) = 0.169967
Iter  8: x = 1.6000, C(x) = 0.070737
Iter  9: x = 1.7000, C(x) = -0.029200
Iter 10: x = 1.8000, C(x) = -0.128844
Iter 11: x = 1.9000, C(x) = -0.227202
Iter 12: x = 2.0000, C(x) = -0.323290
Iter 13: x = 2.1000, C(x) = -0.416147
Iter 14: x = 2.2000, C(x) = -0.504846
Iter 15: x = 2.3000, C(x) = -0.588501
Iter 16: x = 2.4000, C(x) = -0.666276
Iter 17: x = 2.5000, C(x) = -0.737394
Iter 18: x = 2.6000, C(x) = -0.801144
Iter 19: x = 2.7000, C(x) = -0.856889
Iter 20: x = 2.8000, C(x) = -0.904072
Iter 21: x = 2.9000, C(x) = -0.942222
Iter 22: x = 3.0000, C(x) = -0.970958
Iter 23: x = 3.1000, C(x) = -0.989992
Iter 24: x = 3.1000, C(x) = -0.999135
Iter 25: x = 3.1500, C(x) = -0.999135
Iter 26: x = 3.1500, 

# Subsection **9.4.3 Measurement strategies and shot management**

In [14]:
# Hamiltonian: H = 0.5 ZZ + 0.3 ZI + 0.2 XI

def two_qubit_entangled_ansatz(theta: float) -> Circuit:
    qc = Circuit(2)
    qc.ry(theta, 0)
    qc.cx(0, 1)
    return qc

H = [("ZZ", 0.5), ("ZI", 0.3), ("XI", 0.2)]
value = np.pi / 4
expect = expectation_hamiltonian(two_qubit_entangled_ansatz(value), H)
print("Grouped expectation value:", expect)


Grouped expectation value: 0.7121320343559643


# Subsection **9.5.1 Variational Quantum Eigensolver**

In [15]:
# Hamiltonian: H = ZI + IX + 0.5 ZZ
H = [("ZI", 1.0), ("IX", 1.0), ("ZZ", 0.5)]


def vqe_ansatz(params) -> Circuit:
    qc = Circuit(2)
    qc.ry(params[0], 0)
    qc.ry(params[1], 1)
    qc.cx(0, 1)
    qc.ry(params[2], 0)
    qc.ry(params[3], 1)
    return qc


def energy(params) -> float:
    return expectation_hamiltonian(vqe_ansatz(params), H)

params = np.random.uniform(0, 2 * np.pi, 4)
learning_rate = 0.1

for it in range(30):
    grad = np.zeros_like(params)
    eps = 1e-3
    for i in range(len(params)):
        shift = np.zeros_like(params)
        shift[i] = eps
        grad[i] = (energy(params + shift) - energy(params - shift)) / (2 * eps)
    params = params - learning_rate * grad
    print(f"Iter {it:02d}  Energy = {energy(params):.6f}")

print("Optimized energy:", energy(params))


Iter 00  Energy = -0.267711
Iter 01  Energy = -0.398045
Iter 02  Energy = -0.512027
Iter 03  Energy = -0.617137
Iter 04  Energy = -0.719073
Iter 05  Energy = -0.821652
Iter 06  Energy = -0.926993
Iter 07  Energy = -1.035749
Iter 08  Energy = -1.147326
Iter 09  Energy = -1.260102
Iter 10  Energy = -1.371709
Iter 11  Energy = -1.479374
Iter 12  Energy = -1.580341
Iter 13  Energy = -1.672277
Iter 14  Energy = -1.753587
Iter 15  Energy = -1.823559
Iter 16  Energy = -1.882326
Iter 17  Energy = -1.930679
Iter 18  Energy = -1.969817
Iter 19  Energy = -2.001113
Iter 20  Energy = -2.025927
Iter 21  Energy = -2.045497
Iter 22  Energy = -2.060887
Iter 23  Energy = -2.072977
Iter 24  Energy = -2.082475
Iter 25  Energy = -2.089942
Iter 26  Energy = -2.095819
Iter 27  Energy = -2.100449
Iter 28  Energy = -2.104103
Iter 29  Energy = -2.106989
Optimized energy: -2.1069885645207242


# Subsection **9.5.2 Quantum Approximate Optimization Algorithm**

In [21]:
# Triangle graph on 3 vertices: edges (0,1), (1,2), (0,2)
# Cost Hamiltonian: H_C = sum_edges (I - Zi Zj) / 2
H_C = [
    ("III", 1.5),
    ("ZZI", -0.5),
    ("IZZ", -0.5),
    ("ZIZ", -0.5),
]


def qaoa_circuit(gamma: float, beta: float) -> Circuit:
    qc = Circuit(3)
    for q in range(3):
        qc.h(q)
    add_rzz(qc, 2 * gamma, 0, 1)
    add_rzz(qc, 2 * gamma, 1, 2)
    add_rzz(qc, 2 * gamma, 0, 2)
    for q in range(3):
        qc.rx(2 * beta, q)
    return qc


def qaoa_energy(gamma_val: float, beta_val: float) -> float:
    return expectation_hamiltonian(qaoa_circuit(gamma_val, beta_val), H_C)

gamma_vals = np.linspace(0, np.pi, 11)
beta_vals = np.linspace(0, np.pi, 11)

best_E = -1e9
best_params = None

for g in gamma_vals:
    for b in beta_vals:
        E = qaoa_energy(g, b)
        if E > best_E:
            best_E = E
            best_params = (g, b)

print("Best MaxCut value (grid search):", best_E)
print("Best parameters (gamma, beta):", best_params)


Best MaxCut value (grid search): 1.999334805117118
Best parameters (gamma, beta): (np.float64(2.827433388230814), np.float64(1.8849555921538759))


# Subsection **9.5.3 Variational Quantum Linear Solver**

In [22]:
# Define a simple 2x2 Hermitian matrix A = I + 0.5 X
A = np.eye(2, dtype=complex) + 0.5 * qg.X

# Choose a known target solution |x*> = RY(theta_star)|0>
theta_star = np.pi / 3

def vqls_ansatz(theta: float) -> Circuit:
    qc = Circuit(1)
    qc.ry(theta, 0)
    return qc

x_target = vqls_ansatz(theta_star).statevector()

# Define b so that A|x*> = |b>
b_vector = A @ x_target


def residual_cost(theta_val: float) -> float:
    psi = vqls_ansatz(theta_val).statevector()
    residual = A @ psi - b_vector
    return float(np.real(np.vdot(residual, residual)))


theta_vals = np.linspace(0, 2 * np.pi, 100)
cost_vals = [residual_cost(t) for t in theta_vals]

best_idx = np.argmin(cost_vals)

print("Target theta:", theta_star)
print("Best theta:", theta_vals[best_idx])
print("Minimum residual:", cost_vals[best_idx])


Target theta: 1.0471975511965976
Best theta: 1.0789308103237674
Minimum residual: 9.469390776315614e-05


# Subsection **9.6.1 Data encoding circuits: Angle, amplitude, and IQP embeddings**

In [18]:
# Angle encoding for a 3-dimensional vector
x = np.array([0.2, 0.5, -0.1])
qc_angle = Circuit(3)
for j in range(3):
    qc_angle.ry(x[j], j)

# Simple IQP embedding on 2 qubits
z = np.array([0.7, 1.1])
qc_iqp = Circuit(2)
for q in range(2):
    qc_iqp.h(q)
add_rzz(qc_iqp, z[0], 0, 1)
qc_iqp.rz(z[1], 0)

print(qc_angle.draw())
print(qc_iqp.draw())


q0: |0>[ RY]───  ───
q1: |0>───  [ RY]───
q2: |0>───  ───  [ RY]
q0: |0>[ H ]───  [RZZ][ RZ]
q1: |0>───  [ H ][RZZ]───


# Subsection **9.6.2 Quantum Neural Networks and circuit-based models**

In [19]:
def qnn_circuit(x_val, theta_val) -> Circuit:
    qc = Circuit(2)
    # Angle encoding
    qc.ry(x_val[0], 0)
    qc.ry(x_val[1], 1)
    # Simple variational block
    qc.cx(0, 1)
    qc.rz(theta_val[0], 0)
    qc.rx(theta_val[1], 1)
    qc.cx(1, 0)
    qc.ry(theta_val[2], 0)
    return qc


def qnn_forward(x_val, theta_val) -> float:
    return expectation_pauli(qnn_circuit(x_val, theta_val), "ZI")

example_x = np.array([0.2, 0.5])
example_theta = np.array([0.1, 0.3, 0.7])
qc = qnn_circuit(example_x, example_theta)

print(qc.draw())
print("Model output:", qnn_forward(example_x, example_theta))


q0: |0>[ RY]───  ─●─[ RZ]───  ─X─[ RY]
q1: |0>───  [ RY]─X────  [ RX]─●────
Model output: 0.5801801189100396


# Subsection **9.6.3 Training QNNs within hybrid workflows**

In [20]:
# Data (two points for illustration)
X = np.array([[0.2, 0.5],
              [1.0, -0.3]])
y = np.array([+1, -1])


def forward(x_val, theta_val):
    return qnn_forward(x_val, theta_val)


def loss(theta_val):
    preds = np.array([forward(X[i], theta_val) for i in range(len(X))])
    return float(np.mean((preds - y) ** 2))


def train(theta_init, lr=0.2, iters=20):
    theta_val = theta_init.copy()
    for k in range(iters):
        grad = np.zeros_like(theta_val)
        eps = 1e-3
        base = loss(theta_val)
        for i in range(len(theta_val)):
            shifted = theta_val.copy()
            shifted[i] += eps
            grad[i] = (loss(shifted) - base) / eps
        theta_val = theta_val - lr * grad
        print(f"Iter {k:2d}, loss = {base:.6f}")
    return theta_val

theta0 = np.random.uniform(0, 2 * np.pi, 3)
trained = train(theta0)
print("Trained parameters:", trained)


Iter  0, loss = 1.212649
Iter  1, loss = 1.143041
Iter  2, loss = 1.095700
Iter  3, loss = 1.063860
Iter  4, loss = 1.041283
Iter  5, loss = 1.023721
Iter  6, loss = 1.008732
Iter  7, loss = 0.995043
Iter  8, loss = 0.982021
Iter  9, loss = 0.969358
Iter 10, loss = 0.956900
Iter 11, loss = 0.944573
Iter 12, loss = 0.932337
Iter 13, loss = 0.920181
Iter 14, loss = 0.908108
Iter 15, loss = 0.896132
Iter 16, loss = 0.884282
Iter 17, loss = 0.872593
Iter 18, loss = 0.861107
Iter 19, loss = 0.849870
Trained parameters: [2.56913921 1.58490444 6.91099388]
